# Sweep: TOKEN Pooling

Run this notebook in **Colab 3** while running `07_sweep_cls.ipynb` and `07_sweep_mean.ipynb` in separate Colab runtimes.

All results are saved to Google Drive so they can be merged later.


## 1. Setup


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Clone or pull the repository
!git clone https://github.com/TheMattWang/Negation-Origin-Tracing.git 2>/dev/null || (cd Negation-Origin-Tracing && git pull)
%cd Negation-Origin-Tracing


In [ ]:
# Install dependencies
%pip install -q torch lightning transformers datasets pandas pyarrow matplotlib seaborn scikit-learn tqdm


In [ ]:
# Set shared output location on Google Drive
import os
os.environ['DRIVE_OUTPUT'] = '/content/drive/MyDrive/NOT_results'

# Create the directory
!mkdir -p /content/drive/MyDrive/NOT_results

print(f"Results will be saved to: {os.environ['DRIVE_OUTPUT']}")


In [ ]:
# Check GPU
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 2. Download Data


In [ ]:
# Download data if needed
import os
if not os.path.exists('data/raw/train/sst.parquet'):
    print("Downloading data...")
    !python src/data/download.py
else:
    print("Data already exists!")


## 3. Run TOKEN Pooling Sweep

This trains probes on all 6 layers using TOKEN pooling (around "not" token).


In [ ]:
# Run the TOKEN sweep
!chmod +x run_sweep_token.sh
!./run_sweep_token.sh


## 4. Check Results


In [ ]:
import json
import os

results_file = os.path.join(os.environ['DRIVE_OUTPUT'], 'sweep_token', 'results_token.json')

if os.path.exists(results_file):
    with open(results_file, 'r') as f:
        results = json.load(f)
    
    print(f"TOKEN Sweep Results ({len(results)} experiments)")
    print("=" * 50)
    
    for r in sorted(results, key=lambda x: x.get('test_auroc', 0), reverse=True):
        print(f"Layer {r['layer']}: AUROC={r.get('test_auroc', 0):.4f}, Acc={r.get('test_acc', 0):.4f}")
    
    best = max(results, key=lambda x: x.get('test_auroc', 0))
    print(f"\nBest: Layer {best['layer']} with AUROC {best.get('test_auroc', 0):.4f}")
else:
    print(f"Results not found at {results_file}")
